In [ ]:
#Install needed libraries and dependencies
!pip uninstall -y vllm transformers tokenizers
!pip install vllm==0.6.4.post1 transformers==4.46.3 tokenizers==0.20.3 accelerate pandas

Aya-23-8B

In [ ]:
#Tier 3 Voting
import os
import gc
import torch
import pandas as pd
from collections import Counter
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from huggingface_hub import login
from google.colab import drive

print("Mounting Google Drive...")
drive.mount('/content/drive')

from google.colab import userdata
userdata.get('HF_TOKEN')

gc.collect()
torch.cuda.empty_cache()

MODEL_NAME = "alijawad07/aya-23-8B-AWQ-GEMM"

MASTER_CSV_PATH = "PATH_OF_TIER_2_OUTPUT_RATIONALES"
OUTPUT_CSV_PATH = "PATH_OF_OUTPUT"

print("Loading Tokenizer and vLLM Engine...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=userdata.get('HF_TOKEN'))

llm = LLM(
    model=MODEL_NAME,
    quantization="awq",
    dtype="half",
    gpu_memory_utilization=0.90,
    max_model_len=4096,
    enforce_eager=True
)

print("Loading Master Data exclusively from local CSV...")
df = pd.read_csv(MASTER_CSV_PATH)

#Missing column check
required_cols = ['question_en', 'option1', 'option2', 'option3', 'option4', 'target',
                 'rationale_Logical', 'rationale_Balanced', 'rationale_Creative']
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Master CSV is missing the following required columns: {missing_cols}")

target_map = {'option1': 'A', 'option2': 'B', 'option3': 'C', 'option4': 'D'}
df['True_Label'] = df['target'].map(target_map)

print("Formatting prompts for Logical, Balanced, and Creative instances...")
all_prompts = []

for _, row in df.iterrows():
    options_block = f"A. {row['option1']}\nB. {row['option2']}\nC. {row['option3']}\nD. {row['option4']}"

    rationales = [
        row.get('rationale_Logical', "No rationale provided."),
        row.get('rationale_Balanced', "No rationale provided."),
        row.get('rationale_Creative', "No rationale provided.")
    ]

#Prompt
    for rationale in rationales:
        messages = [
            {"role": "system", "content": "You are an expert test-taking AI. Use the provided rationale to answer the question. Respond ONLY with the single letter of the correct option (A, B, C, or D)."},
            {"role": "user", "content": f"### RATIONALE\n{rationale}\n\n### QUESTION\n{row['question_en']}\n\n### OPTIONS\n{options_block}\n\nThe correct option is:"}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        all_prompts.append(text)

params = SamplingParams(
    temperature=0.0,
    max_tokens=1,
    logprobs=20
)

print(f"\nRunning vLLM Log-Likelihood Extraction for {len(all_prompts)} total prompts...")
outputs = llm.generate(all_prompts, params)

#LogProb
print("\nCalculating Majority Votes and Accuracies...")
results_log = []
correct_count = 0
valid_choices = ['A', 'B', 'C', 'D']

def get_best_letter(logprobs_dict):
    best_letter = "C"
    highest_prob = -float('inf')

    for token_id, logprob_obj in logprobs_dict.items():
        token_str = logprob_obj.decoded_token.strip().upper()
        if token_str in valid_choices and logprob_obj.logprob > highest_prob:
            highest_prob = logprob_obj.logprob
            best_letter = token_str

    return best_letter

for i in range(len(df)):
    row = df.iloc[i]
    truth = row['True_Label']

    out_logical = outputs[i * 3]
    out_balanced = outputs[i * 3 + 1]
    out_creative = outputs[i * 3 + 2]

    logprobs_logical = out_logical.outputs[0].logprobs[0]
    logprobs_balanced = out_balanced.outputs[0].logprobs[0]
    logprobs_creative = out_creative.outputs[0].logprobs[0]

    pred_logical = get_best_letter(logprobs_logical)
    pred_balanced = get_best_letter(logprobs_balanced)
    pred_creative = get_best_letter(logprobs_creative)

    preds = [pred_logical, pred_balanced, pred_creative]

    counts = Counter(preds)
    max_count = max(counts.values())
    tied_options = [opt for opt, count in counts.items() if count == max_count]

#Tiebreaker
    if len(tied_options) == 1:
        final_pred = tied_options[0]
    else:
        final_pred = pred_logical

    is_correct = (final_pred == truth)
    if is_correct:
        correct_count += 1

    results_log.append({
        "Question_ID": i,
        "True_Label": truth,
        "Pred_Logical": pred_logical,
        "Pred_Balanced": pred_balanced,
        "Pred_Creative": pred_creative,
        "Final_Voted_Pred": final_pred,
        "Is_Correct": is_correct
    })

accuracy = (correct_count / len(df)) * 100
print("\nENSEMBLE VOTING COMPLETE!")
print(f"FINAL ACCURACY: {accuracy:.2f}%")

final_df = pd.DataFrame(results_log)
final_df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Voting breakdown saved to:\n{OUTPUT_CSV_PATH}")